# 📡 Passo 3 v2 — STA/LTA Corrigido
**Correções aplicadas:**
- Loop duplicado removido
- Caminhos agnósticos de SO (funciona no Windows e Mac)
- Score contínuo para ROC corrigido
- Análise de sensibilidade funcional


## 1. Configuração

In [ ]:
import os, json, time, warnings
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal as scipy_signal
warnings.filterwarnings('ignore')

# ── DETECTA O SO E AJUSTA CAMINHOS AUTOMATICAMENTE ──────────────
import platform
SO = platform.system()  # 'Windows' ou 'Darwin' (Mac)

if SO == 'Windows':
    PASTA_RAIZ    = r"C:\Users\vish8\OneDrive\Documentos\TCC\data\scedc-pds"
    PASTA_PROJETO = r"C:\Users\vish8\OneDrive\Documentos\TCC\Trabalho\artefacts"
else:  # Mac
    PASTA_RAIZ    = "/Users/alvarosamp/Documents/Projetos/TCC/data/scedc-pds"
    PASTA_PROJETO = "/Users/alvarosamp/Documents/Projetos/TCC/Trabalho/artefacts"

PASTA_XML     = os.path.join(PASTA_RAIZ, "FDSNstationXML")
INVENTARIO    = os.path.join(PASTA_PROJETO, "data", "inventario_dados.csv")
PASTA_RESULTS = os.path.join(PASTA_PROJETO, "results")
os.makedirs(PASTA_RESULTS, exist_ok=True)
os.makedirs(os.path.join(PASTA_PROJETO, "figures"), exist_ok=True)

# ── PARÂMETROS DO PIPELINE ────────────────────────────────────────
SR_ALVO     = 40.0
CANAL_ALVO  = "BHZ"
OUTPUT_RESP = "VEL"
PRE_FILT    = (0.5, 1.0, 18.0, 20.0)
WATER_LEVEL = 60
FREQ_MIN    = 0.5
FREQ_MAX    = 15.0

# ── PARÂMETROS STA/LTA ────────────────────────────────────────────
STA_SEG   = 1.0
LTA_SEG   = 10.0
THRESH_ON = 3.0
THRESH_OFF= 1.5
TOLERANCIA_SEG = 15.0

print(f"✅ Sistema operacional : {SO}")
print(f"   PASTA_RAIZ         : {PASTA_RAIZ}")
print(f"   PASTA_PROJETO      : {PASTA_PROJETO}")
print(f"   STA={STA_SEG}s | LTA={LTA_SEG}s | thr={THRESH_ON}/{THRESH_OFF}")


## 2. Funções

In [ ]:
from obspy import read, read_inventory
import pandas as pd

def encontrar_xml(rede, estacao, pasta_xml):
    for root, _, files in os.walk(pasta_xml):
        for f in files:
            if f.endswith('.xml') and rede in f and estacao in f:
                return os.path.join(root, f)
    return None

def processar_trace_raw(tr_input, caminho_xml):
    try:
        tr = tr_input.copy()
        if tr.stats.sampling_rate != SR_ALVO:
            tr.resample(SR_ALVO)
        tr.detrend('linear')
        tr.detrend('demean')
        tr.taper(max_percentage=0.05, type='cosine')
        inv = read_inventory(caminho_xml)
        tr.remove_response(inventory=inv, output=OUTPUT_RESP,
                           pre_filt=PRE_FILT, water_level=WATER_LEVEL)
        tr.filter('bandpass', freqmin=FREQ_MIN, freqmax=FREQ_MAX, zerophase=True)
        dados = tr.data.astype(np.float32)
        return None if dados.std() < 1e-15 else dados
    except:
        return None

def sta_lta_eficiente(sinal, sr, sta_seg, lta_seg):
    nSTA    = int(sta_seg * sr)
    nLTA    = int(lta_seg * sr)
    energia = sinal ** 2
    k_sta   = np.ones(nSTA) / nSTA
    k_lta   = np.ones(nLTA) / nLTA
    sta     = np.convolve(energia, k_sta, mode='full')[:len(sinal)]
    lta     = np.convolve(energia, k_lta, mode='full')[:len(sinal)]
    return np.where(lta > 1e-20, sta / lta, 0.0)

def detectar_eventos(cf, thresh_on, thresh_off, sr, min_dur=1.0):
    min_n   = int(min_dur * sr)
    eventos = []
    dentro  = False
    t0      = 0
    for t, val in enumerate(cf):
        if not dentro and val > thresh_on:
            dentro = True; t0 = t
        elif dentro and val < thresh_off:
            dentro = False
            if (t - t0) >= min_n:
                eventos.append((t0 / sr, t / sr))
    if dentro and (len(cf) - t0) >= min_n:
        eventos.append((t0 / sr, len(cf) / sr))
    return eventos

print("✅ Funções definidas")


## 3. Carregar inventário

In [ ]:
df_inv = pd.read_csv(INVENTARIO)
df_uso = df_inv[
    (df_inv['xml_disponivel'] == True) &
    (df_inv['razao_var'] > 10)
].copy().reset_index(drop=True)

# ── FIX: corrigir caminhos no inventário para o SO atual ─────────
def corrigir_caminho(caminho_original):
    """Reconstrói o caminho para o SO atual usando o nome do arquivo."""
    nome = os.path.basename(str(caminho_original))
    # Busca recursiva pelo arquivo
    for root, _, files in os.walk(PASTA_RAIZ):
        if nome in files:
            return os.path.join(root, nome)
    return caminho_original

df_uso['caminho'] = df_uso['caminho'].apply(corrigir_caminho)

print(f"Arquivos para avaliar: {len(df_uso)}")
print(f"Exemplo de caminho  : {df_uso['caminho'].iloc[0]}")

# Verificar se os caminhos existem
n_ok = df_uso['caminho'].apply(os.path.exists).sum()
print(f"Caminhos válidos    : {n_ok}/{len(df_uso)}")


## 4. Visualização em um arquivo

In [ ]:
row = df_uso.loc[df_uso['razao_var'].idxmax()]
print(f"Arquivo: {row['arquivo']} | razão_var: {row['razao_var']:.0f}x")

st  = read(row['caminho'])
xml = encontrar_xml(row['rede'], row['estacao'], PASTA_XML)
tr  = next((t for t in st if t.stats.channel.endswith(CANAL_ALVO)), None)
dados = processar_trace_raw(tr, xml) if tr and xml else None

if dados is not None:
    tempo = np.arange(len(dados)) / SR_ALVO
    cf    = sta_lta_eficiente(dados, SR_ALVO, STA_SEG, LTA_SEG)
    evs   = detectar_eventos(cf, THRESH_ON, THRESH_OFF, SR_ALVO)

    fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

    ax = axes[0]
    ax.plot(tempo, dados, 'steelblue', lw=0.6)
    ax.axvline(row['t_evento_est'], color='red', ls='--', lw=1.5,
               label=f"t_real={row['t_evento_est']}s")
    for t0e, t1e in evs:
        ax.axvspan(t0e, t1e, alpha=0.2, color='orange')
    ax.set_ylabel('Velocidade (m/s)')
    ax.set_title(f'Sinal + Detecções STA/LTA — {row["arquivo"]}')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

    ax = axes[1]
    ax.plot(tempo, cf, 'darkorange', lw=1.0)
    ax.axhline(THRESH_ON,  color='red',   ls='--', lw=1.2, label=f'ON={THRESH_ON}')
    ax.axhline(THRESH_OFF, color='green', ls='--', lw=1.2, label=f'OFF={THRESH_OFF}')
    ax.axvline(row['t_evento_est'], color='red', ls=':', lw=1.2, alpha=0.7)
    ax.set_ylim([0, min(cf.max()*1.1, 25)])
    ax.set_ylabel('CF = STA/LTA')
    ax.set_xlabel('Tempo (s)')
    ax.set_title('Characteristic Function')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

    plt.tight_layout()
    fig_path = os.path.join(PASTA_PROJETO, "figures", "passo3_cf_visual.png")
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.show()

    print(f"Eventos detectados: {len(evs)}")
    for i,(t0e,t1e) in enumerate(evs):
        print(f"  {i+1}: {t0e:.1f}s → {t1e:.1f}s")
    print(f"Evento real: {row['t_evento_est']}s")


## 5. Avaliação sistemática (loop corrigido)

In [ ]:
# ── FIX: processar cada arquivo UMA VEZ (sem duplicação) ─────────
resultados = []
all_scores = []   # (score_continuo, label) para curva ROC
tempo_total = 0.0
n_amostras_total = 0

print(f"Avaliando {len(df_uso)} arquivos...")
print("-" * 65)

for _, row in df_uso.iterrows():   # iterrows() correto — 1 linha por arquivo
    try:
        if not os.path.exists(row['caminho']):
            print(f"  ⚠️  {row['arquivo']} — caminho não encontrado")
            continue

        st  = read(row['caminho'])
        xml = encontrar_xml(row['rede'], row['estacao'], PASTA_XML)
        if xml is None:
            continue

        tr = next((t for t in st if t.stats.channel.endswith(CANAL_ALVO)), None)
        if tr is None:
            continue

        dados = processar_trace_raw(tr, xml)
        if dados is None:
            continue

        # Inferência com medição de tempo
        t_ini = time.perf_counter()
        cf    = sta_lta_eficiente(dados, SR_ALVO, STA_SEG, LTA_SEG)
        evs   = detectar_eventos(cf, THRESH_ON, THRESH_OFF, SR_ALVO)
        tempo_total     += time.perf_counter() - t_ini
        n_amostras_total += len(dados)

        # Ground truth
        t_real    = row['t_evento_est']
        detectado = any(
            abs(t0e - t_real) <= TOLERANCIA_SEG or (t0e <= t_real <= t1e)
            for t0e, t1e in evs
        )

        # ── Score contínuo para ROC ──────────────────────────────
        # Score = CF máxima em todo o sinal normalizado pelo percentil 95
        # Isso garante que o score reflita o "pico relativo" da CF
        cf_norm = cf / (np.percentile(cf, 95) + 1e-10)

        # Score do evento: CF máxima próxima ao t_real
        idx_ini = max(0, int((t_real - TOLERANCIA_SEG) * SR_ALVO))
        idx_fim = min(len(cf_norm), int((t_real + TOLERANCIA_SEG) * SR_ALVO))
        score_ev = float(cf_norm[idx_ini:idx_fim].max()) if idx_fim > idx_ini else 0.0

        # Score do ruído: CF máxima fora da janela do evento
        mask_ruido = np.ones(len(cf_norm), dtype=bool)
        mask_ruido[max(0,idx_ini-20):min(len(cf_norm),idx_fim+20)] = False
        score_ruid = float(cf_norm[mask_ruido].max()) if mask_ruido.any() else 0.0

        all_scores.append((score_ruid, 0))   # ruído
        all_scores.append((score_ev,   1))   # evento

        status = "✅ VP" if detectado else "❌ FN"
        print(f"  {status}  {row['arquivo']:25s}  "
              f"CF_max={score_ev*np.percentile(cf,95):6.2f}  "
              f"alarmes={len(evs):2d}  t_real={t_real:.0f}s")

        resultados.append({
            'arquivo'  : row['arquivo'],
            'razao_var': float(row['razao_var']),
            't_real'   : float(t_real),
            'detectado': bool(detectado),
            'n_alarmes': len(evs),
            'score'    : score_ev,
        })

    except Exception as e:
        print(f"  ❌ ERRO {row['arquivo']}: {e}")

# ── Métricas ──────────────────────────────────────────────────────
from sklearn.metrics import roc_auc_score, roc_curve, f1_score

df_res = pd.DataFrame(resultados)
vp     = df_res['detectado'].sum()
fn     = len(df_res) - vp

precisao = 1.0   # 1 evento por arquivo, sem FP por arquivo
recall   = vp / max(len(df_res), 1)
f1       = 2 * precisao * recall / max(precisao + recall, 1e-10)

scores_v = [s for s,_ in all_scores]
labels_v = [l for _,l in all_scores]
try:
    auc = roc_auc_score(labels_v, scores_v)
except:
    auc = 0.0

dur_s    = n_amostras_total / SR_ALVO
ms_por_s = (tempo_total / dur_s) * 1000 if dur_s > 0 else 0

print()
print("=" * 65)
print(f"RESULTADOS STA/LTA")
print(f"  Arquivos avaliados : {len(df_res)}")
print(f"  VP (detectados)    : {vp}")
print(f"  FN (perdidos)      : {fn}")
print(f"  Precisão           : {precisao:.4f}")
print(f"  Recall             : {recall:.4f}")
print(f"  F1-score           : {f1:.4f}")
print(f"  AUC-ROC            : {auc:.4f}")
print(f"  Tempo inferência   : {ms_por_s:.4f} ms/s de sinal")


## 6. Análise de sensibilidade

In [ ]:
def avaliar_params(df_uso, sta, lta, thr):
    """Avalia STA/LTA com parâmetros dados. Retorna (f1, auc)."""
    dets, sc, lb = [], [], []
    for _, row in df_uso.iterrows():
        try:
            if not os.path.exists(row['caminho']): continue
            st  = read(row['caminho'])
            xml = encontrar_xml(row['rede'], row['estacao'], PASTA_XML)
            if xml is None: continue
            tr  = next((t for t in st if t.stats.channel.endswith(CANAL_ALVO)), None)
            if tr is None: continue
            dados = processar_trace_raw(tr, xml)
            if dados is None: continue

            cf      = sta_lta_eficiente(dados, SR_ALVO, sta, lta)
            evs     = detectar_eventos(cf, thr, thr*0.5, SR_ALVO)
            t_real  = row['t_evento_est']
            det     = any(
                abs(t0e-t_real)<=TOLERANCIA_SEG or (t0e<=t_real<=t1e)
                for t0e,t1e in evs
            )
            dets.append(int(det))

            cf_n   = cf / (np.percentile(cf, 95) + 1e-10)
            i0     = max(0, int((t_real-TOLERANCIA_SEG)*SR_ALVO))
            i1     = min(len(cf_n), int((t_real+TOLERANCIA_SEG)*SR_ALVO))
            sc.append(float(cf_n[i0:i1].max()) if i1>i0 else 0.0); lb.append(1)
            mask   = np.ones(len(cf_n), bool)
            mask[max(0,i0-20):min(len(cf_n),i1+20)] = False
            sc.append(float(cf_n[mask].max()) if mask.any() else 0.0); lb.append(0)
        except: continue

    if not dets: return 0.0, 0.0
    rec = sum(dets)/len(dets)
    f   = 2*rec/(1+rec) if rec>0 else 0.0
    try: a = roc_auc_score(lb, sc)
    except: a = 0.0
    return f, a

print("Análise de sensibilidade:")
print("=" * 55)

stas   = [0.5, 1.0, 2.0, 3.0, 5.0]
ltas   = [5.0, 10.0, 15.0, 20.0, 30.0]
thrs   = [1.5, 2.0, 3.0, 4.0, 5.0, 7.0, 10.0]

f1_sta, f1_lta, f1_thr = [], [], []

print("\nVariando STA (LTA=10s, thr=3.0):")
for s in stas:
    f,a = avaliar_params(df_uso, s, 10.0, 3.0)
    f1_sta.append(f)
    print(f"  STA={s:4.1f}s  F1={f:.3f}  AUC={a:.3f}")

print("\nVariando LTA (STA=1s, thr=3.0):")
for l in ltas:
    f,a = avaliar_params(df_uso, 1.0, l, 3.0)
    f1_lta.append(f)
    print(f"  LTA={l:5.1f}s  F1={f:.3f}  AUC={a:.3f}")

print("\nVariando threshold (STA=1s, LTA=10s):")
for t in thrs:
    f,a = avaliar_params(df_uso, 1.0, 10.0, t)
    f1_thr.append(f)
    print(f"  thr={t:5.1f}   F1={f:.3f}  AUC={a:.3f}")

# ── Plot ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, x, y, xlabel, color in [
    (axes[0], stas, f1_sta, 'STA (s)',       'steelblue'),
    (axes[1], ltas, f1_lta, 'LTA (s)',       'darkorange'),
    (axes[2], thrs, f1_thr, 'Threshold ON',  'crimson'),
]:
    ax.plot(x, y, color=color, marker='o', lw=2)
    ax.set_xlabel(xlabel); ax.set_ylabel('F1-score')
    ax.set_ylim([0, 1.05]); ax.grid(True, alpha=0.3)
    ax.set_title(f'Sensibilidade ao {xlabel}')

plt.suptitle('STA/LTA — Análise de Sensibilidade', fontsize=11)
plt.tight_layout()
fig_path = os.path.join(PASTA_PROJETO, "figures", "passo3_sensibilidade.png")
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()

best_sta = stas[np.argmax(f1_sta)]
best_lta = ltas[np.argmax(f1_lta)]
best_thr = thrs[np.argmax(f1_thr)]
print(f"\nMelhores: STA={best_sta}s | LTA={best_lta}s | thr={best_thr}")


## 7. Curva ROC e tabela final

In [ ]:
# Recalcular com melhores parâmetros
scores_f, labels_f = [], []
dets_f = []

for _, row in df_uso.iterrows():
    try:
        if not os.path.exists(row['caminho']): continue
        st  = read(row['caminho'])
        xml = encontrar_xml(row['rede'], row['estacao'], PASTA_XML)
        if xml is None: continue
        tr  = next((t for t in st if t.stats.channel.endswith(CANAL_ALVO)), None)
        if tr is None: continue
        dados = processar_trace_raw(tr, xml)
        if dados is None: continue

        cf    = sta_lta_eficiente(dados, SR_ALVO, best_sta, best_lta)
        evs   = detectar_eventos(cf, best_thr, best_thr*0.5, SR_ALVO)
        t_r   = row['t_evento_est']
        det   = any(abs(t0e-t_r)<=TOLERANCIA_SEG or (t0e<=t_r<=t1e) for t0e,t1e in evs)
        dets_f.append(int(det))

        cf_n = cf / (np.percentile(cf,95)+1e-10)
        i0   = max(0, int((t_r-TOLERANCIA_SEG)*SR_ALVO))
        i1   = min(len(cf_n), int((t_r+TOLERANCIA_SEG)*SR_ALVO))
        scores_f.append(float(cf_n[i0:i1].max()) if i1>i0 else 0.0); labels_f.append(1)
        mask = np.ones(len(cf_n), bool)
        mask[max(0,i0-20):min(len(cf_n),i1+20)] = False
        scores_f.append(float(cf_n[mask].max()) if mask.any() else 0.0); labels_f.append(0)
    except: continue

rec_f = sum(dets_f)/max(len(dets_f),1)
f1_f  = 2*rec_f/(1+rec_f) if rec_f>0 else 0.0
try:    auc_f = roc_auc_score(labels_f, scores_f)
except: auc_f = 0.0

fpr, tpr, thr_roc = roc_curve(labels_f, scores_f)
f1_roc = [f1_score(labels_f, (np.array(scores_f)>=t).astype(int), zero_division=0)
          for t in thr_roc]
idx_bf  = np.argmax(f1_roc)
best_f1 = f1_roc[idx_bf]
best_t  = thr_roc[idx_bf]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(fpr, tpr, 'steelblue', lw=2, label=f'STA/LTA (AUC={auc_f:.3f})')
axes[0].plot([0,1],[0,1],'k--',lw=1,alpha=0.5,label='Aleatório (0.500)')
axes[0].scatter(fpr[idx_bf],tpr[idx_bf],color='red',s=100,zorder=5,
                label=f'Best F1={best_f1:.3f}')
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR (Recall)')
axes[0].set_title('Curva ROC — STA/LTA')
axes[0].legend(fontsize=9); axes[0].grid(True,alpha=0.3)

axes[1].plot(thr_roc, f1_roc, 'crimson', lw=2)
axes[1].axvline(best_t, color='red', ls='--', lw=1.5, label=f'thr_ótimo={best_t:.2f}')
axes[1].set_xlabel('Threshold (score normalizado)')
axes[1].set_ylabel('F1-score')
axes[1].set_title('F1 × Threshold')
axes[1].legend(fontsize=9); axes[1].grid(True,alpha=0.3)

plt.tight_layout()
fig_path = os.path.join(PASTA_PROJETO, "figures", "passo3_roc.png")
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()

# ── Tabela final ──────────────────────────────────────────────────
print("=" * 65)
print("PRIMEIRA LINHA DA TABELA COMPARATIVA — STA/LTA")
print("=" * 65)
print(f"  F1-score     : {f1_f:.4f}")
print(f"  AUC-ROC      : {auc_f:.4f}")
print(f"  Tamanho      : N/A (algoritmo)")
print(f"  Inferência   : {ms_por_s:.4f} ms/s de sinal")
print()
print(f"  {'Método':<15} {'F1':>8} {'AUC':>8} {'Tamanho':>10} {'ms/s':>10}")
print(f"  {'-'*15} {'-'*8} {'-'*8} {'-'*10} {'-'*10}")
print(f"  {'STA/LTA':<15} {f1_f:>8.4f} {auc_f:>8.4f} {'N/A':>10} {ms_por_s:>10.4f}")
print(f"  {'Dense AE':<15} {'—':>8} {'—':>8} {'—':>10} {'—':>10}")
print(f"  {'CNN 1D AE':<15} {'—':>8} {'—':>8} {'—':>10} {'—':>10}")
print(f"  {'LSTM AE':<15} {'—':>8} {'—':>8} {'—':>10} {'—':>10}")

resultado = {
    "metodo"       : "STA/LTA",
    "versao"       : "2.0",
    "parametros"   : {"sta_seg":best_sta,"lta_seg":best_lta,"threshold":float(best_thr)},
    "f1"           : float(f1_f),
    "auc"          : float(auc_f),
    "tamanho_kb"   : None,
    "inferencia_ms": float(ms_por_s),
}
result_path = os.path.join(PASTA_RESULTS, "resultado_staltalvta_v2.json")
with open(result_path,'w') as f:
    json.dump(resultado, f, indent=2)
print(f"\n💾 Resultado salvo: {result_path}")
print("\n🚀 Passo 3 concluído → Próximo: Passo 4 — Dense Autoencoder")
